# Задание 1. CustomResNet

Требуется самостоятельно реализовать свёрточную сеть на базе архитектуры ResNet-18, описанной в [Deep Residual Learning for Image Recognition 🎓[arxiv]](https://arxiv.org/abs/1512.03385) (для выполнения задания достаточно ознакомиться с содержанием **раздела 4.2**).

Создайте свою собственную сеть на основе архитектуры ResNet, описанной в лекции. От 15 до 25 слоев.

Используйте заготовки классов `CustomResnet`, `BasicBlock`.

**Важно:** Не допускается целиком копировать код из [исходников PyTorch 🐾[git]](https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py).

**Важно:** Если вы используете готовый фрагмент кода, то должны быть приведены ссылка на источник и комментарии.


## Формат результата

* Графики loss и accuracy при обучении (можно использовать TensorBoard)

<img src = "https://edunet.kea.su/repo/EduNet-web_dependencies/dev-2.4/Exercises/EX08/result_4_task_ex08.png" width="1000">

* Значение accuracy на тесте должно составить не менее 0.77
* Вывод: Сравнительный анализ результатов точности вашей модели и библиотечной версии.

Импорт необходимых библиотек:

In [2]:
!pip install torchmetrics
!pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 38.5 MB/s eta 0:00:00


In [3]:
import timm
import torch
import torchmetrics
import torch.nn as nn
import lightning as L

from torchsummary import summary
from torchvision.models import resnet18
from torch.utils.data import DataLoader
from torchvision import models, datasets
from torchvision.transforms import v2

## Структура модели

Прежде чем создавать собственную сеть, посмотрите архитектуру ResNet-18. Для разнообразия посмотрим на "Зоопарк моделей" PyTorch. Для этого используйте пакет [torchsummary 🛠️[doc]](https://pypi.org/project/torch-summary/) или метод `add_graph` [🛠️[doc]](https://pytorch.org/docs/stable/tensorboard.html?highlight=add_graph#torch.utils.tensorboard.writer.SummaryWriter.add_graph) для TensorBoard.

In [4]:
from torchsummary import summary
from torchvision.models import resnet18
import torch

model = resnet18()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
summary(model, (3, 32, 32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 16, 16]           9,408
       BatchNorm2d-2           [-1, 64, 16, 16]             128
              ReLU-3           [-1, 64, 16, 16]               0
         MaxPool2d-4             [-1, 64, 8, 8]               0
            Conv2d-5             [-1, 64, 8, 8]          36,864
       BatchNorm2d-6             [-1, 64, 8, 8]             128
              ReLU-7             [-1, 64, 8, 8]               0
            Conv2d-8             [-1, 64, 8, 8]          36,864
       BatchNorm2d-9             [-1, 64, 8, 8]             128
             ReLU-10             [-1, 64, 8, 8]               0
       BasicBlock-11             [-1, 64, 8, 8]               0
           Conv2d-12             [-1, 64, 8, 8]          36,864
      BatchNorm2d-13             [-1, 64, 8, 8]             128
             ReLU-14             [-1, 6

Визуализировав библиотечную модель, можно заметить, что пространственные размеры тензора на выходе из последнего сверточного блока 7×7. Если подавать в такую сеть картинки, которые в 7 раз меньше (CIFAR-10 вместо ImageNet), то этот тензор схлопнется в вектор.
 Из этого можно сделать вывод о необходимости отключить слои, которые содержат свертки с большим шагом в начале сети.

## Загрузка данных

Блок кода, отвечающий за загрузку данных (можно использовать без внесения изменений)

In [5]:
# Load and preprocess the data. Don't change this code

# https://github.com/facebookarchive/fb.resnet.torch/issues/180
cifar10_mean = (0.491, 0.482, 0.447)
cifar10_std = (0.247, 0.244, 0.262)


# Data preprocessing
transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(cifar10_mean, cifar10_std),
    ]
)

cifar_train = datasets.CIFAR10(
    "content", train=True, transform=transform, download=True
)
train_set, val_set = torch.utils.data.random_split(cifar_train, [45000, 5000])

100%|██████████| 170M/170M [00:13<00:00, 13.1MB/s]


Рекомендуется делать всю отладку на небольших фрагментах датасета, для чего создать объекты класса `Subset` и соответствующие `DataLoader`-объекты.


```python
mini_trainset , _ = torch.utils.data.random_split(trainset, [5000, 45000])
mini_testset, _  = torch.utils.data.random_split(testset, [1000, 9000])
...

```

In [6]:
train_set

In [7]:
mini_trainset , _ = torch.utils.data.random_split(train_set, [5000, 40000])
mini_testset, _  = torch.utils.data.random_split(val_set, [1000, 4000])

## Блок кода для обучения

В него можно вносить изменения, как минимум менять гиперпараметры и добавить код для тестирования.

In [8]:
class LightningModel(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)
        self.valid_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters())
        return optimizer

    def training_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        loss = self.criterion(out, y)
        self.train_acc.update(out, y)
        self.log("loss/train", loss.detach().item())
        return loss

    def on_train_epoch_end(self):
        self.log("accuracy/train", self.train_acc.compute(), prog_bar=True)
        self.train_acc.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        self.valid_acc.update(out, y)

    def on_validation_epoch_end(self):
        self.log("accuracy/val", self.valid_acc.compute(), prog_bar=True)
        self.valid_acc.reset()

## Основная часть задания

Создайте свою собственную сеть на основе архитектуры ResNet, описанной в лекции. От 15  до 25 слоев.

Используйте заготовки классов `CustomResnet`, `BasicBlock`.

Прежде чем приступать к работе, просмотрите оригинальную [статью 🎓[arxiv]](https://arxiv.org/pdf/1512.03385.pdf) (для выполнения задания достаточно ознакомиться с содержанием **раздела 4.2**).

Не допускается копировать код из [исходников PyTorch 🐾[git]](https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py).

Если вы используете готовый фрагмент кода, то должна быть приведена ссылка на источник и комментарии.

Цель — добиться точности > 0.77.
При разумной архитектуре и гиперпараметрах для этого достаточно 20 эпох.

In [13]:
class CustomResnet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64,  64, stride=1)  # 32x32
        self.layer2 = self._make_layer(64,  128, stride=2)  # 16x16
        self.layer3 = self._make_layer(128, 256, stride=2)  # 8x8

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, in_channels, out_channels, stride=1):
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        return BasicBlock(in_channels, out_channels, stride=stride, downsample=downsample)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        init_obj = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            init_obj = self.downsample(x)
        out += init_obj
        out = self.relu(out)
        return out

## Обучите вашу модель на CIFAR-10

Не забудьте вернуть в датасет данные, если вы удаляли их для ускорения отладки.

Оптимизатор, количество эпох, шаг обучения, критерий останова выберите на свое усмотрение.

Цель — добиться точности > 0.77.
При разумной архитектуре и гиперпараметрах для этого достаточно 20 эпох (примерно 6 минут).

Рекомендация: сохраняйте лучшую модель, считайте метрики на лучшей модели.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/ex_4/lightning_logs --port 6009

In [14]:
train_loader = DataLoader(train_set, batch_size=128, num_workers=2, shuffle=True)
val_loader = DataLoader(val_set, batch_size=128, num_workers=2, shuffle=False)

In [15]:
from lightning.pytorch.loggers import TensorBoardLogger

model = CustomResnet()

max_epochs = 20

lightning_model = LightningModel(model)

exp_name = "my_resnet"
trainer = L.Trainer(
    max_epochs=max_epochs,
    logger=TensorBoardLogger(save_dir="/content/ex_4/lightning_logs", name=exp_name),
    callbacks=[
        L.pytorch.callbacks.ModelCheckpoint(
            monitor="accuracy/val",
            mode="max",
            filename="best",
        )
    ],
    log_every_n_steps=1
)

trainer.fit(lightning_model, train_loader, val_loader)


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ CustomResnet       │  1.2 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 2 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ valid_acc │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 1.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.2 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 33                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


Проверьте качество на тестовой части датасета.

In [18]:
cifar_test = datasets.CIFAR10(
  "content", train=False, transform=transform, download=True
)
test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False)

# Your code here
best_model = CustomResnet(num_classes=10)
best_ckpt_path = trainer.checkpoint_callback.best_model_path
lightning_best = LightningModel.load_from_checkpoint(best_ckpt_path, model=best_model)
best_model = lightning_best.model.to(device)
best_model.eval()

test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

with torch.no_grad():
  for x, y in test_loader:
      x, y = x.to(device), y.to(device)
      out = best_model(x)
      test_acc.update(out, y)

print(f"CustomResnet — Test accuracy: {test_acc.compute():.4f}")

CustomResnet — Test accuracy: 0.8054


## Обучите ResNet-18

Теперь обучите ResNet-18  из `torchvision.models` на том же CIFAR-10.
Используйте непредобученную модель (`weights = None`). Для обучения на CIFAR-10 потребуется изменить параметры линейного слоя на выходе модели, так как в этом датасете 10 классов, а не 1000, как в ImageNet.


После завершения обучения сравните точность вашей модели с точностью ResNet-18  из `torchvision.models`, а также с точностью, полученной авторами статьи на CIFAR-10:


<img src = "https://edunet.kea.su/repo/EduNet-web_dependencies/dev-2.4/Exercises/EX08/resnet_accuracy.png" width="600">

Замените последний слой, отвечающий за классификацию, линейным слоем с 10-ю выходами.

In [19]:
from torchvision.models import resnet18

resnet_orig = resnet18(weights=None)
resnet_orig.fc = nn.Linear(resnet_orig.fc.in_features, 10)


Обучите модель (достаточно 10 эпох — примерно 3 минуты).

In [21]:
# Your code here
lightning_resnet = LightningModel(resnet_orig)

exp_name = "resnet_orig"
trainer_resnet = L.Trainer(
    max_epochs=10,
    logger=TensorBoardLogger(save_dir="/content/ex_4/lightning_logs", name=exp_name),
    callbacks=[
        L.pytorch.callbacks.ModelCheckpoint(
            monitor="accuracy/val",
            mode="max",
            filename="best",
        )
    ],
    log_every_n_steps=1
)

trainer_resnet.fit(lightning_resnet, train_loader, val_loader)


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ResNet             │ 11.2 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss   │      0 │ train │     0 │
│ 2 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ valid_acc │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 71                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Оцените точность библиотечной модели на test:

In [22]:
# Your code here
best_resnet = resnet18(weights=None)
best_resnet.fc = nn.Linear(best_resnet.fc.in_features, 10)
best_ckpt_resnet = trainer_resnet.checkpoint_callback.best_model_path
lightning_best_resnet = LightningModel.load_from_checkpoint(best_ckpt_resnet, model=best_resnet)
best_resnet = lightning_best_resnet.model.to(device)
best_resnet.eval()

test_acc_resnet = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = best_resnet(x)
        test_acc_resnet.update(out, y)

print(f"resnet_orig—test accuracy:{test_acc_resnet.compute():.4f}")


resnet_orig—test accuracy:0.7535

**Напишите вывод:**

Сравнительный анализ результатов точности вашей модели и библиотечной версии.
